# 📊 Benchmark Analysis Grid — ALO vs GA vs Random Search

## Permutation Flow Shop Scheduling — Taillard Instances

### 🔗 Dependencies

This notebook **depends on** `ALO_FlowShop.ipynb` for all algorithm implementations and the core `tai20_5` data. It loads them via `%run`.

### 📋 What This Grid Covers

| Phase | Content |
|---|---|
| **Phase 3** | 30 independent runs × 3 algorithms on `tai20_5` |
| **Phase 4** | Multiple Taillard instances (20–100 jobs, 5–20 machines) |
| **Statistics** | Best, Mean, Worst, Std Dev, Avg Time |
| **Visuals** | Convergence curves, bar charts, box plots, scalability grids |

In [ ]:
# =============================================================================
# Cell — Load Notebook 1 (ALO_FlowShop.ipynb)
# =============================================================================
# This imports all functions, data, and algorithm implementations
# from the main ALO Flow Shop notebook.

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
import warnings
warnings.filterwarnings('ignore')

# Run the first notebook to get all definitions
get_ipython().run_line_magic('run', './ALO_FlowShop.ipynb')

print("\n" + "=" * 60)
print("✅ Loaded all algorithms from ALO_FlowShop.ipynb")
print(f"   Instance: {n_jobs} jobs × {n_machines} machines")
print("=" * 60)

## 🧪 Phase 3 — 30 Independent Runs (tai20_5)

Each algorithm runs 30× with seeds `run×100+42`. Results are collected into a stats grid.

In [ ]:
# =============================================================================
# Cell — run_multiple_trials() — Multi-Run Harness
# =============================================================================

def run_multiple_trials(algorithm_fn, processing_times, n_runs=30,
                        algorithm_name="Algorithm", **kwargs):
    """
    Run an algorithm n_runs times with seeds run×100+42.
    Returns a stats dict with all_makespans, all_times, best/avg/worst/std.
    """
    all_makespans = []
    all_times = []
    all_convergences = []
    best_overall_schedule = None
    best_overall_makespan = np.inf

    print(f"\n{'─'*50}")
    print(f"  {algorithm_name} — {n_runs} runs")
    print(f"{'─'*50}")

    for run in range(n_runs):
        np.random.seed(run * 100 + 42)
        t0 = time.time()
        result = algorithm_fn(processing_times, **kwargs)
        elapsed = time.time() - t0

        mk = result['best_makespan']
        all_makespans.append(mk)
        all_times.append(elapsed)

        if 'convergence' in result:
            all_convergences.append(result['convergence'])

        if mk < best_overall_makespan:
            best_overall_makespan = mk
            best_overall_schedule = result.get('best_schedule', None)

        if (run + 1) % 5 == 0 or run == 0:
            print(f"    Run {run+1:2d}/{n_runs}: makespan={mk:.0f}  ({elapsed:.2f}s)")

    arr = np.array(all_makespans)
    return {
        'all_makespans': arr,
        'all_times': np.array(all_times),
        'best_schedule': best_overall_schedule,
        'best_makespan': best_overall_makespan,
        'avg_makespan': np.mean(arr),
        'worst_makespan': np.max(arr),
        'std_makespan': np.std(arr, ddof=1),
        'avg_time': np.mean(all_times),
        'all_convergences': all_convergences
    }

print("✅ run_multiple_trials() defined")

In [ ]:
# =============================================================================
# Cell — Execute 30 Runs × 3 Algorithms (tai20_5)
# =============================================================================

print("=" * 60)
print("PHASE 3 — 30 INDEPENDENT RUNS")
print("Instance: tai20_5 (20 jobs × 5 machines)")
print("=" * 60)

# --- 1. Random Search ---
rs_multi = run_multiple_trials(
    lambda pt, **kw: {'best_makespan': random_search(pt, n_iter=5000)[1],
                      'best_schedule': random_search(pt, n_iter=5000)[0],
                      'execution_time': 0},
    processing_times, n_runs=30, algorithm_name="Random Search"
)

# --- 2. Genetic Algorithm ---
ga_multi = run_multiple_trials(
    genetic_algorithm, processing_times, n_runs=30,
    algorithm_name="Genetic Algorithm",
    pop_size=40, max_iter=200, crossover_rate=0.8, mutation_rate=0.1, elitism=2
)

# --- 3. ALO ---
alo_multi = run_multiple_trials(
    alo, processing_times, n_runs=30,
    algorithm_name="ALO",
    pop_size=40, max_iter=200, lb=-4.0, ub=4.0, verbose=False
)

print("\n✅ All 90 runs complete!")

## 📊 Statistics Grid — tai20_5

| Algorithm | Best | Mean | Worst | Std Dev | Avg Time (s) |
|---|---|---|---|---|---|
| Random Search | | | | | |
| Genetic Algorithm | | | | | |
| ALO | | | | | |

In [ ]:
# =============================================================================
# Cell — Print Statistics Grid
# =============================================================================

def print_stats_grid(results_dict, instance_name="tai20_5"):
    """Print a formatted statistics grid."""
    print(f"\n{'='*72}")
    print(f"  📊 STATISTICS GRID — {instance_name}")
    print(f"{'='*72}")
    print(f"  {'Algorithm':<20} {'Best':<8} {'Mean':<10} {'Worst':<8} {'Std':<10} {'Time(s)':<8}")
    print(f"  {'─'*68}")
    for nm, r in results_dict.items():
        print(f"  {nm:<20} {r['best_makespan']:<8.0f} {r['avg_makespan']:<10.1f} "
              f"{r['worst_makespan']:<8.0f} {r['std_makespan']:<10.2f} {r['avg_time']:<8.2f}")
    print(f"  {'='*72}")

    best_alg = min(results_dict.items(), key=lambda x: x[1]['best_makespan'])
    stable_alg = min(results_dict.items(), key=lambda x: x[1]['std_makespan'])
    fast_alg = min(results_dict.items(), key=lambda x: x[1]['avg_time'])

    print(f"\n  🏆  Best makespan:  {best_alg[0]} ({best_alg[1]['best_makespan']:.0f})")
    print(f"  📉  Most stable:    {stable_alg[0]} (σ={stable_alg[1]['std_makespan']:.2f})")
    print(f"  ⚡  Fastest:        {fast_alg[0]} ({fast_alg[1]['avg_time']:.2f}s)")

results_tai20_5 = {
    'Random Search':      rs_multi,
    'Genetic Algorithm':  ga_multi,
    'ALO':                alo_multi
}
print_stats_grid(results_tai20_5, "tai20_5 (20×5)")

## 📈 Convergence Grid — Mean ± 1σ Across 30 Runs

In [ ]:
# =============================================================================
# Cell — Convergence Curves (Mean ± Std)
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

# ALO
alo_c = np.array(alo_multi['all_convergences'])
alo_m, alo_s = np.mean(alo_c, axis=0), np.std(alo_c, axis=0, ddof=1)
its = np.arange(len(alo_m))
ax.plot(its, alo_m, '#2c3e50', lw=2.5, label='ALO')
ax.fill_between(its, alo_m - alo_s, alo_m + alo_s, color='#2c3e50', alpha=0.15)

# GA
ga_c = np.array(ga_multi['all_convergences'])
ga_m, ga_s = np.mean(ga_c, axis=0), np.std(ga_c, axis=0, ddof=1)
ax.plot(its, ga_m, '#e74c3c', lw=2.5, label='GA')
ax.fill_between(its, ga_m - ga_s, ga_m + ga_s, color='#e74c3c', alpha=0.15)

# RS baselines
ax.axhline(rs_multi['best_makespan'], color='#27ae60', ls='--', lw=1.5, alpha=0.7,
           label=f"RS Best={rs_multi['best_makespan']:.0f}")
ax.axhline(rs_multi['avg_makespan'], color='#27ae60', ls=':', lw=1.5, alpha=0.5,
           label=f"RS Mean={rs_multi['avg_makespan']:.1f}")

ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax.set_ylabel('Makespan', fontsize=12, fontweight='bold')
ax.set_title('Convergence — Mean ± 1σ (30 runs, tai20_5)', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Final — ALO: {alo_m[-1]:.1f}±{alo_s[-1]:.2f}  |  GA: {ga_m[-1]:.1f}±{ga_s[-1]:.2f}")

## 📊 Visual Grid — Bar Charts + Box Plot

In [ ]:
# =============================================================================
# Cell — Comparison Bar Charts + Box Plot
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

algs = ['Random Search', 'Genetic Algorithm', 'ALO']
cols = ['#27ae60', '#e74c3c', '#2c3e50']
res = [rs_multi, ga_multi, alo_multi]

# 1. Best makespan
ax = axes[0]
vals = [r['best_makespan'] for r in res]
bars = ax.bar(algs, vals, color=cols, alpha=0.85, edgecolor='white')
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+3, f'{v:.0f}',
            ha='center', fontweight='bold', fontsize=11)
ax.set_title('Best Makespan', fontweight='bold'); ax.grid(axis='y', alpha=0.3)

# 2. Average ± std
ax = axes[1]
means = [r['avg_makespan'] for r in res]
stds  = [r['std_makespan'] for r in res]
xb = np.arange(3)
bars = ax.bar(xb, means, yerr=stds, color=cols, alpha=0.85, capsize=6, edgecolor='white')
for b, v in zip(bars, means):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+5, f'{v:.1f}',
            ha='center', fontweight='bold', fontsize=11)
ax.set_xticks(xb); ax.set_xticklabels(algs)
ax.set_title('Avg Makespan ± 1σ', fontweight='bold'); ax.grid(axis='y', alpha=0.3)

# 3. Execution time
ax = axes[2]
times = [r['avg_time'] for r in res]
bars = ax.bar(algs, times, color=cols, alpha=0.85, edgecolor='white')
for b, v in zip(bars, times):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+max(times)*0.02,
            f'{v:.2f}s', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Avg Execution Time', fontweight='bold'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# ── Box Plot ──
fig, ax = plt.subplots(figsize=(10, 5))
data = [r['all_makespans'] for r in res]
bp = ax.boxplot(data, labels=algs, patch_artist=True, widths=0.5,
                showmeans=True, meanline=True)
for patch, c in zip(bp['boxes'], cols):
    patch.set_facecolor(c); patch.set_alpha(0.7)
for m in bp['medians']: m.set_color('black'); m.set_linewidth(2)
for m in bp['means']:   m.set_color('gold'); m.set_linewidth(2)

for i, (d, c) in enumerate(zip(data, cols)):
    jit = np.random.normal(0, 0.04, len(d))
    ax.scatter(np.full_like(d, i+1)+jit, d, alpha=0.35, color=c, s=25, zorder=5)

ax.set_title('Makespan Distribution (30 runs)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.legend([mpatches.Patch(color='black'), plt.Line2D([0],[0], color='gold', lw=2)],
          ['Median', 'Mean'], fontsize=10)
plt.tight_layout()
plt.show()

print("✅ Visual grid complete")

## 🏭 Phase 4 — Multiple Taillard Instances

| Instance | Jobs × Machines | Category |
|---|---|---|
| `tai20_5` | 20 × 5 | Small |
| `tai20_10` | 20 × 10 | Small |
| `tai50_5` | 50 × 5 | Medium |
| `tai50_10` | 50 × 10 | Medium |
| `tai100_10` | 100 × 10 | Large |
| `tai100_20` | 100 × 20 | Large |

In [ ]:
# =============================================================================
# Cell — Define All Benchmark Instances
# =============================================================================

# tai20_5 is already loaded as `processing_times` from notebook 1

tai20_10_data = np.array([
    [13,65,89,53,86,91,36,72,26,33],[12,95,89,67,59,24,59,23,94,82],
    [29,78,74,86,79,26,62,39,44,36],[45,59,23,87,89,46,96,75,14,63],
    [35,87,56,68,39,44,19,63,85,77],[68,95,11,49,60,21,99,63,89,64],
    [85,83,40,31,49,14,52,33,77,89],[49,60,68,10,86,73,69,44,11,44],
    [21,92,15,69,54,71,87,64,63,83],[81,21,74,53,99,39,54,46,27,55],
    [42,97,87,89,64,66,68,61,9,82],[16,85,98,95,88,29,98,34,19,88],
    [79,84,72,89,45,31,62,63,48,10],[30,3,56,66,48,26,71,72,82,43],
    [17,14,97,18,87,89,57,41,95,79],[84,55,45,80,65,79,27,40,92,28],
    [33,16,12,73,39,21,58,63,39,61],[73,29,61,19,35,77,99,56,85,73],
    [31,64,79,76,29,68,12,93,60,63],[44,11,90,78,48,89,30,57,68,69]
])

tai50_5_data = np.array([
    [76,51,60,63,26],[59,98,5,76,44],[74,76,25,91,46],[12,27,65,96,20],
    [42,38,79,23,7],[38,69,37,41,63],[64,60,56,47,10],[98,58,55,44,93],
    [27,17,95,74,27],[69,35,42,49,34],[37,11,49,83,20],[36,18,83,31,98],
    [79,34,41,80,31],[29,70,85,18,23],[32,63,92,19,95],[72,54,55,46,52],
    [26,65,87,48,31],[50,56,38,6,83],[11,25,95,76,92],[98,63,22,53,85],
    [99,53,41,17,68],[87,21,23,79,20],[31,67,13,8,95],[40,45,59,14,25],
    [31,78,80,69,56],[73,26,17,17,41],[30,98,68,56,46],[99,84,31,30,19],
    [76,76,76,78,56],[39,66,53,17,90],[12,28,28,43,28],[99,46,32,7,52],
    [12,55,30,62,44],[56,59,36,52,84],[75,94,81,37,57],[56,39,14,90,52],
    [96,57,26,52,96],[12,35,90,4,52],[10,16,66,54,35],[58,83,7,19,91],
    [20,64,61,65,52],[12,17,22,36,51],[14,60,98,64,12],[73,60,90,48,57],
    [28,91,84,55,63],[79,35,87,73,44],[42,62,80,81,84],[57,64,83,11,86],
    [79,30,97,58,46],[68,16,43,75,48]
])

tai50_10_data = np.array([
    [98,58,74,90,91,67,88,18,57,92],[79,22,89,69,42,42,40,95,38,33],
    [15,57,99,92,71,30,82,53,57,23],[16,32,94,73,81,90,80,61,48,56],
    [92,57,76,16,28,62,42,54,19,98],[32,41,70,42,16,65,21,22,22,37],
    [97,88,14,87,56,50,10,8,80,68],[44,12,64,87,69,83,85,72,8,13],
    [40,89,7,73,84,5,49,91,93,26],[54,38,87,37,43,1,63,48,1,99],
    [55,40,55,90,33,78,90,52,90,94],[91,24,56,35,27,66,60,93,18,66],
    [26,87,87,11,23,27,89,88,9,22],[16,36,25,83,80,76,23,50,13,72],
    [19,94,40,36,27,32,73,69,74,20],[69,97,38,85,95,55,87,25,26,51],
    [57,55,18,12,14,57,17,62,43,34],[88,66,25,88,36,12,2,88,90,97],
    [91,64,62,63,58,50,20,32,93,84],[31,85,40,37,43,76,87,75,60,85],
    [79,74,28,79,57,47,58,82,13,68],[57,41,43,28,67,80,93,81,96,31],
    [56,66,54,96,15,98,79,7,64,28],[17,9,70,26,49,99,51,33,69,99],
    [23,43,17,87,93,37,42,89,10,90],[50,31,47,75,26,30,26,93,93,88],
    [95,87,38,44,14,84,79,9,85,61],[82,12,95,6,17,91,51,84,43,99],
    [82,40,57,86,9,92,73,33,94,42],[50,91,51,59,71,12,47,50,66,47],
    [48,49,87,4,33,60,35,98,54,96],[80,85,47,67,35,88,59,16,79,40],
    [96,52,40,36,76,56,73,37,69,69],[44,40,95,31,99,98,42,79,60,61],
    [94,36,84,94,80,78,21,11,79,72],[61,15,73,42,90,94,48,94,64,72],
    [95,95,55,84,92,96,29,45,80,91],[62,33,69,51,58,20,86,25,14,75],
    [15,85,51,87,12,81,88,68,56,85],[28,15,82,56,96,96,33,71,92,56],
    [21,28,63,18,86,23,38,42,17,13],[36,71,9,18,82,21,34,44,96,72],
    [95,61,36,52,24,64,20,42,20,63],[83,29,99,78,65,32,12,74,57,26],
    [24,89,27,99,73,25,44,25,80,12],[51,75,68,35,60,58,72,63,54,39],
    [13,63,66,17,96,75,56,24,43,23],[87,72,93,86,97,83,15,26,30,72],
    [79,39,54,90,64,75,80,51,37,55],[36,14,84,30,51,52,57,30,87,72]
])

# Dictionary for all instances (tai20_5 uses `processing_times` from notebook 1)
all_instances = {
    'tai20_5':   {'data': processing_times, 'jobs': 20,  'machines': 5},
    'tai20_10':  {'data': tai20_10_data,    'jobs': 20,  'machines': 10},
    'tai50_5':   {'data': tai50_5_data,     'jobs': 50,  'machines': 5},
    'tai50_10':  {'data': tai50_10_data,    'jobs': 50,  'machines': 10},
}

print("✅ Instances defined:")
for n, i in all_instances.items():
    print(f"   {n}: {i['jobs']}j×{i['machines']}m — {i['data'].shape}")

In [ ]:
# =============================================================================
# Cell — Multi-Instance Benchmark Runner
# =============================================================================

def evaluate_all(instances, n_runs=5, ga_pop=40, ga_iter=200,
                 alo_pop=40, alo_iter=200, rs_samples=5000):
    """Run all 3 algorithms on each instance. Returns {name: {alg: stats}}."""
    results = {}
    for name, info in instances.items():
        pt = info['data']
        nj, nm = info['jobs'], info['machines']
        print(f"\n{'#'*60}\n# {name} ({nj}j×{nm}m)\n{'#'*60}")

        rs = run_multiple_trials(
            lambda p, **kw: {'best_makespan': random_search(p, n_iter=rs_samples)[1],
                             'best_schedule': random_search(p, n_iter=rs_samples)[0],
                             'execution_time': 0},
            pt, n_runs=n_runs, algorithm_name=f"RS-{name}")
        ga = run_multiple_trials(
            genetic_algorithm, pt, n_runs=n_runs, algorithm_name=f"GA-{name}",
            pop_size=ga_pop, max_iter=ga_iter, crossover_rate=0.8, mutation_rate=0.1, elitism=2)
        al = run_multiple_trials(
            alo, pt, n_runs=n_runs, algorithm_name=f"ALO-{name}",
            pop_size=alo_pop, max_iter=alo_iter, lb=-4.0, ub=4.0, verbose=False)

        results[name] = {'Random Search': rs, 'Genetic Algorithm': ga, 'ALO': al}
    return results


def print_benchmark_grid(all_results):
    """Print the full benchmark comparison grid."""
    print(f"\n{'='*105}")
    print(f"  📊 BENCHMARK COMPARISON GRID")
    print(f"{'='*105}")
    hdr = f"  {'Instance':<10} {'Algorithm':<18} {'Best':<8} {'Mean':<10} {'Std':<8} {'Time(s)':<8}"
    print(hdr)
    print(f"  {'─'*95}")
    for iname, ires in all_results.items():
        for idx, (aname, r) in enumerate(ires.items()):
            pfx = iname if idx == 0 else ''
            print(f"  {pfx:<10} {aname:<18} {r['best_makespan']:<8.0f} "
                  f"{r['avg_makespan']:<10.1f} {r['std_makespan']:<8.2f} {r['avg_time']:<8.2f}")
        print(f"  {'─'*95}")

    print(f"\n{'='*105}")
    print("  🏆  BEST PER INSTANCE")
    print(f"{'='*105}")
    for iname, ires in all_results.items():
        best = min(ires.items(), key=lambda x: x[1]['best_makespan'])
        stable = min(ires.items(), key=lambda x: x[1]['std_makespan'])
        fast = min(ires.items(), key=lambda x: x[1]['avg_time'])
        print(f"  {iname:<10}  Best={best[0]:<18}({best[1]['best_makespan']:.0f})  "
              f"Stable={stable[0]:<10}(σ={stable[1]['std_makespan']:.2f})  "
              f"Fast={fast[0]:<10}({fast[1]['avg_time']:.2f}s)")

print("✅ evaluate_all() + print_benchmark_grid() defined")

In [ ]:
# =============================================================================
# Cell — Execute Phase 4 Benchmark
# =============================================================================

N_RUNS = 5  # Use 30 for full analysis
instances_to_run = {k: v for k, v in all_instances.items()}

print(f"PHASE 4 — {N_RUNS} runs × 3 algorithms × {len(instances_to_run)} instances\n")

all_results = evaluate_all(
    instances_to_run, n_runs=N_RUNS,
    ga_pop=40, ga_iter=200, alo_pop=40, alo_iter=200, rs_samples=5000
)

print_benchmark_grid(all_results)

## 📈 Scalability Grid — Best Makespan & Time Across Instances

In [ ]:
# =============================================================================
# Cell — Scalability Visualization
# =============================================================================

names = list(all_results.keys())
x = np.arange(len(names))
w = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Best makespan
ax = axes[0]
for j, (alg, clr) in enumerate([('Random Search','#27ae60'),('Genetic Algorithm','#e74c3c'),('ALO','#2c3e50')]):
    vals = [all_results[n][alg]['best_makespan'] for n in names]
    ax.bar(x + (j-1)*w, vals, w, label=alg, color=clr, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_title('Best Makespan Across Instances', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

# Execution time
ax = axes[1]
for j, (alg, clr) in enumerate([('Random Search','#27ae60'),('Genetic Algorithm','#e74c3c'),('ALO','#2c3e50')]):
    vals = [all_results[n][alg]['avg_time'] for n in names]
    ax.bar(x + (j-1)*w, vals, w, label=alg, color=clr, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_title('Avg Execution Time Across Instances', fontweight='bold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 🔍 Analysis Summary

| # | Question |
|---|---|
| 1 | Which algorithm achieved the **best makespan**? |
| 2 | Which algorithm was **most stable** (lowest σ)? |
| 3 | Which algorithm was **fastest**? |
| 4 | How does **ALO scale** with problem size? |

In [ ]:
# =============================================================================
# Cell — Automated Analysis
# =============================================================================

print("=" * 65)
print("  🔍 ANALYSIS — PER INSTANCE WINNERS")
print("=" * 65)

for iname, ires in all_results.items():
    info = all_instances[iname]
    best = min(ires.items(), key=lambda x: x[1]['best_makespan'])
    stable = min(ires.items(), key=lambda x: x[1]['std_makespan'])
    fast = min(ires.items(), key=lambda x: x[1]['avg_time'])
    print(f"\n  📦 {iname} ({info['jobs']}j×{info['machines']}m)")
    print(f"     🏆  Best:     {best[0]:<20} ({best[1]['best_makespan']:.0f})")
    print(f"     📉  Stable:   {stable[0]:<20} (σ={stable[1]['std_makespan']:.2f})")
    print(f"     ⚡  Fast:     {fast[0]:<20} ({fast[1]['avg_time']:.2f}s)")

print(f"\n{'='*65}")
print("  📈 SCALABILITY — ALO vs GA vs RS")
print("="*65)
for iname in sorted(all_results.keys(), key=lambda n: all_instances[n]['jobs']*all_instances[n]['machines']):
    info = all_instances[iname]
    al = all_results[iname]['ALO']
    ga = all_results[iname]['Genetic Algorithm']
    rs = all_results[iname]['Random Search']
    print(f"\n  {iname:<10} ({info['jobs']}j×{info['machines']}m)")
    print(f"     ALO mean={al['avg_makespan']:.1f}  vs RS={rs['avg_makespan']:.1f}  vs GA={ga['avg_makespan']:.1f}")
    print(f"     ALO time={al['avg_time']:.2f}s  GA time={ga['avg_time']:.2f}s")
    impr_rs = (rs['avg_makespan']-al['avg_makespan'])/rs['avg_makespan']*100
    impr_ga = (ga['avg_makespan']-al['avg_makespan'])/ga['avg_makespan']*100
    print(f"     ALO vs RS: {impr_rs:+.2f}%  |  ALO vs GA: {impr_ga:+.2f}%")

print(f"\n{'='*65}")
print("  ✅ Analysis complete")
print("="*65)